In [1]:
import numpy as np
from scipy.integrate import solve_ivp
import qme 
import importlib
from scipy.linalg import expm

## Deriving the Lindblad master equation

The Lindblad master equation can be derived from a microscopic model. We start with the system \(S\) and its environment \(E\).

The Hamiltonian of the combined system is

$$
H=H_S+H_E+H_{SE},
\qquad
H_{SE}=\sum_\alpha A_\alpha\otimes B_\alpha.
$$

Here, \(H_S\) acts on the system, \(H_E\) acts on the environment, and \(H_{SE}\) describes their interaction. The operator \(A_\alpha\) acts on \(S\), while \(B_\alpha\) acts on \(E\).

For the **full system** \(S+E\), the state evolves according to

$$
i\frac{d\rho_{SE}}{dt}=[H,\rho_{SE}].
$$

### Interaction picture

Define \(H_0=H_S+H_E\). The interaction-picture state is

$$
\rho_{SE}^{(I)}(t)
=e^{iH_0t}\rho_{SE}(t)e^{-iH_0t},
$$

and the interaction-picture coupling is

$$
H_{SE}^{(I)}(t)
=e^{iH_0t}H_{SE}e^{-iH_0t}.
$$

The equation of motion becomes

$$
\boxed{
\frac{d\rho_{SE}^{(I)}(t)}{dt}
=-i[H_{SE}^{(I)}(t),\rho_{SE}^{(I)}(t)]
}
\tag{1}
$$

Integrate both sides from \(0\) to \(t\):

$$\boxed{\rho_{SE}^{(I)}(t)=\rho_{SE}^{(I)}(0) -i\int_0^t d\tau\, [H_{SE}^{(I)}(\tau),\rho_{SE}^{(I)}(\tau)]}\tag{2}$$

### Trace over the environment

We want an equation for the system state

$$
\rho_S^{(I)}(t)
=\operatorname{Tr}_E\!\left[\rho_{SE}^{(I)}(t)\right].
$$

Taking the partial trace of equation (1) gives

$$
\frac{d\rho_S^{(I)}(t)}{dt}
=-i\operatorname{Tr}_E
\!\left[
H_{SE}^{(I)}(t),\rho_{SE}^{(I)}(t)
\right].
\tag{3}
$$

Now insert equation (2) into the right-hand side:

$$
\boxed{
\begin{aligned}
\frac{d\rho_S^{(I)}(t)}{dt}
={}&-i\operatorname{Tr}_E
\!\left[
H_{SE}^{(I)}(t),\rho_{SE}^{(I)}(0)
\right]\\[2mm]
&-\int_0^t d\tau\,
\operatorname{Tr}_E
\!\left[
H_{SE}^{(I)}(t),
\left[
H_{SE}^{(I)}(\tau),
\rho_{SE}^{(I)}(\tau)
\right]
\right].
\end{aligned}
}
\tag{4}
$$

Equation (4) is **exact**, but it is not yet an equation for \(\rho_S^{(I)}\) alone: its right-hand side still contains the joint state \(\rho_{SE}^{(I)}(\tau)\). We need further approximations before obtaining a closed master equation for the system.

In [2]:
# Basis: |0>, |1> for both the system S and environment E
zero = np.array([1, 0], dtype=complex)
one = np.array([0, 1], dtype=complex)
I = np.eye(2, dtype=complex)

sigma_minus = np.outer(zero, one.conj())
sigma_plus = sigma_minus.conj().T

# Example: the system and environment exchange an excitation
coupling = 0.4
H_SE = coupling * (
    np.kron(sigma_plus, sigma_minus)
    + np.kron(sigma_minus, sigma_plus)
)

# Start with S excited and E in |0>
psi0 = np.kron(one, zero)
rho_SE_0 = np.outer(psi0, psi0.conj())

times = np.linspace(0, 3, 301)

solution = solve_ivp(
    qme.utils.joint_rhs,
    (times[0], times[-1]),
    rho_SE_0.reshape(16),
    args=(H_SE,),            
    t_eval=times,
    rtol=1e-11,
    atol=1e-13,
)

if not solution.success:
    raise RuntimeError(solution.message)

rho_SE_t = solution.y.T.reshape(-1, 4, 4)
rho_S_t = np.array([qme.utils.trace_environment(rho) for rho in rho_SE_t])

# Check equation (3) at one time:
# d(rho_S)/dt = -i Tr_E([H_SE, rho_SE]).
index = 100
t = times[index]

derivative_from_equation = qme.utils.trace_environment(
    -1j * (H_SE @ rho_SE_t[index] - rho_SE_t[index] @ H_SE)
)

# Estimate the derivative directly from nearby numerical states
dt = times[index + 1] - times[index - 1]
derivative_from_states = (
    rho_S_t[index + 1] - rho_S_t[index - 1]
) / dt

print("Time:", t)
print("\nDerivative from equation (3):\n", derivative_from_equation)
print("\nDerivative from nearby states:\n", derivative_from_states)
print(
    "\nMaximum difference:",
    np.max(np.abs(derivative_from_equation - derivative_from_states))
)

Time: 1.0

Derivative from equation (3):
 [[ 0.28694244+0.j  0.        +0.j]
 [ 0.        +0.j -0.28694244+0.j]]

Derivative from nearby states:
 [[ 0.28693938+0.j  0.        +0.j]
 [ 0.        +0.j -0.28693938+0.j]]

Maximum difference: 3.060659541587185e-06


### Born approximation

We now make an approximation for the joint system–environment state appearing in the exact equation. Assume that the interaction is weak and that the environment stays close to a fixed state $\rho_E$:

$$
\rho_{SE}^{(I)}(t)
\approx \rho_S^{(I)}(t)\otimes\rho_E,
\qquad
\rho_E(t)\approx\rho_E.
$$

This is called the **Born approximation**. The environment state $\rho_E$ might, for example, be a thermal equilibrium state. We also assume

$$
\operatorname{Tr}_E\!\left[B_\alpha^{(I)}(t)\rho_E\right]=0
$$

for the environmental operators in $H_{SE}^{(I)}$. With this choice, the first-order interaction term vanishes. A nonzero average can instead be absorbed into an effective system Hamiltonian.

### Markov approximation

The environment has a short **memory time**: its correlations decay quickly compared with the time over which the system changes. In the integral, we therefore approximate

$$
\rho_S^{(I)}(\tau)\approx\rho_S^{(I)}(t).
$$

This is the **Markov approximation**. It says that the system state inside the memory integral can be replaced by its state at the present time.

Using these assumptions in the equation from the previous page gives

$$
\boxed{
\frac{d\rho_S^{(I)}(t)}{dt}
=
-\int_0^t d\tau\,
\operatorname{Tr}_E
\left[
H_{SE}^{(I)}(t),
\left[
H_{SE}^{(I)}(\tau),
\rho_S^{(I)}(t)\otimes\rho_E
\right]
\right].
}
$$

This is the **Born–Redfield master equation** in the interaction picture. The right-hand side now depends on $\rho_S^{(I)}(t)$ rather than the unknown joint state $\rho_{SE}^{(I)}(\tau)$, so the equation is closed for the system.

### Extending the memory integral

The upper limit is still $t$. If environmental correlations decay rapidly, contributions from the distant past are negligible. Writing $s=t-\tau$, one can extend the upper limit of the **memory-time** integral to infinity:

$$
\boxed{
\frac{d\rho_S^{(I)}(t)}{dt}
\approx
-\int_0^\infty ds\,
\operatorname{Tr}_E
\left[
H_{SE}^{(I)}(t),
\left[
H_{SE}^{(I)}(t-s),
\rho_S^{(I)}(t)\otimes\rho_E
\right]
\right].
}
$$

The environmental memory is described by correlation functions such as

$$
C_{\alpha\beta}(s)
=
\operatorname{Tr}_E
\left[
B_\alpha^{(I)}(s)B_\beta^{(I)}(0)\rho_E
\right].
$$

When $C_{\alpha\beta}(s)$ decays rapidly, large values of $s$ contribute very little to the integral.

### Why this is not yet the Lindblad equation

The Redfield equation preserves the trace and Hermiticity, but its solutions are **not guaranteed to remain positive** under every choice of parameters and approximations.

To reach the standard Lindblad form, we usually make a further **secular approximation**: rapidly oscillating terms that average to zero are removed. The resulting coefficients are calculated from the environment correlation functions. Under the appropriate weak-coupling and Markov assumptions, this gives a master equation whose evolution is completely positive and trace preserving.

### Typical examples

- Dynamics of electronic excitations in a solid coupled to phonon modes of the crystal (bath).
- Coupling of atoms to modes of the electromagnetic field (quantum optics) $\rightarrow$ Wigner–Weisskopf theory of spontaneous emission.

## 4) Solving the master equation / properties of the Liouvillian

- Cast the Lindblad master equation into standard first-order ODE system form (flatten $\rho$ into a vector):

$$
\dot{\rho}=\mathcal{L}\rho,
\qquad
\rho=\sum_{m,n}\rho_{mn}|m\rangle\langle n|.
$$

  $\mathcal{L}$ is the **Liouvillian** (a Lindblad superoperator). For a Hilbert space of dimension $d$, the vectorized density matrix has dimension $d^2$. For $N$ spin-$\frac12$ particles, its dimension is $2^{2N}$.

- Formal solution:

$$
\rho(t)=e^{\mathcal{L}t}\rho(0)
\qquad
\bigl(e^{\mathcal{L}t}\text{ is a CPTP map}\bigr).
$$

- $\mathcal{L}$ is not Hermitian $\rightarrow$ it has complex eigenvalues.

  Trace preservation and physical evolution $\rightarrow$
  $\operatorname{Re}(\lambda)\leq 0$ for all eigenvalues: modes can decay rather than grow.

- There is at least one state with

$$
\mathcal{L}\rho_{\mathrm{ss}}=0,
$$

  called a **steady state**, since

$$
\rho(t)=e^{\mathcal{L}t}\rho(0)
\underset{t\to\infty}{\longrightarrow}
\rho_{\mathrm{ss}}
$$

  when that steady state is the unique attracting state.

- The **Liouvillian gap** is the smallest nonzero decay rate:

$$
\Delta_{\mathcal{L}}
=
\min_{\operatorname{Re}(\lambda)<0}
\left|\operatorname{Re}(\lambda)\right|.
$$

  It sets a timescale on which $\rho_{\mathrm{ss}}$ is approached.

- Closing of the Liouvillian gap $\rightarrow$ “dissipative phase transition”; properties of $\rho_{\mathrm{ss}}$ change.

In [3]:
# Basis order: |g>, |e>
g = np.array([1, 0], dtype=complex)
e = np.array([0, 1], dtype=complex)

Pg = np.outer(g, g.conj())
Pe = np.outer(e, e.conj())
sigma_minus = np.outer(g, e.conj())

omega = 2.0
gamma = 0.8

H = omega * Pe
Gamma = np.sqrt(gamma) * sigma_minus
rho0 = Pe.copy()

# The Liouvillian is a 4 × 4 matrix because rho is 2 × 2.
L_matrix = qme.utils.liouvillian_matrix(H, [Gamma])

print("Liouvillian matrix:\n", np.round(L_matrix, 4))
print("\nEigenvalues:", np.round(np.linalg.eigvals(L_matrix), 4))

# Formal solution: vec(rho(t)) = exp(t L) vec(rho(0))
t = 2.0
rho_t = (expm(t * L_matrix) @ rho0.reshape(4)).reshape(2, 2)

# Your analytical result
rho_exact = (1 - np.exp(-gamma * t)) * Pg + np.exp(-gamma * t) * Pe

print("\nNumerical rho(t):\n", np.round(rho_t, 6))
print("\nAnalytical rho(t):\n", np.round(rho_exact, 6))
print("\nSolutions agree:", np.allclose(rho_t, rho_exact))

# The ground state is stationary: L[rho_ss] = 0.
rho_ss = Pg
steady_state_error = L_matrix @ rho_ss.reshape(4)

print("\nSteady state rho_ss:\n", rho_ss)
print("L[rho_ss] = 0:", np.allclose(steady_state_error, 0))

# Slowest nonzero decay rate
eigenvalues = np.linalg.eigvals(L_matrix)
decay_rates = -eigenvalues.real
positive_rates = decay_rates[decay_rates > 1e-10]

gap = np.min(positive_rates)
print("\nLiouvillian gap:", gap)
print("Relaxation timescale 1/gap:", 1 / gap)

Liouvillian matrix:
 [[ 0. +0.j  0. +0.j  0. +0.j  0.8+0.j]
 [ 0. +0.j -0.4+2.j  0. +0.j  0. +0.j]
 [ 0. +0.j  0. +0.j -0.4-2.j  0. +0.j]
 [ 0. +0.j  0. +0.j  0. +0.j -0.8+0.j]]

Eigenvalues: [ 0. +0.j -0.4+2.j -0.4-2.j -0.8+0.j]

Numerical rho(t):
 [[0.798103+0.j 0.      +0.j]
 [0.      +0.j 0.201897+0.j]]

Analytical rho(t):
 [[0.798103+0.j 0.      +0.j]
 [0.      +0.j 0.201897+0.j]]

Solutions agree: True

Steady state rho_ss:
 [[1.+0.j 0.+0.j]
 [0.+0.j 0.+0.j]]
L[rho_ss] = 0: True

Liouvillian gap: 0.39999999999999997
Relaxation timescale 1/gap: 2.5
